# Ollama Transfer + Eval Notebook

Use this notebook after the tuned GGUF/model has been registered in Ollama. It checks that the local Ollama call path is not adding thinking text or extra formatting, then runs the same `eval/harness.py` used by the CLI.

Recommended order:
1. Set `MODEL_NAME` to your Ollama tag.
2. Run the Ollama/template cells.
3. Run the format-transfer smoke test.
4. Run the smoke eval.
5. Flip `RUN_FULL_EVAL = True` only when the smoke looks clean.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import textwrap
import time

try:
    import pandas as pd
except Exception:
    pd = None

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "eval" / "harness.py").exists() and (p / "prompts" / "litmus_generation_prompt.md").exists():
            return p
    raise RuntimeError("Could not find repo root. Run this notebook from inside the slm repo.")

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "eval"))
print("repo:", ROOT)
print("python:", sys.executable)

## Configure Models

`USE_JUDGE` requires `PROMPTLENS_API_KEY` in the environment, unless you edit `JUDGE_CFG` to another provider. With `USE_JUDGE = False`, the notebook still checks formatting and programmatic gates, but the verdict is not a real expert-quality eval.

In [ ]:
MODEL_NAME = "qwen3-apush:latest"       # change this to the tuned Ollama tag
BASE_MODEL_NAME = "qwen3:4b"            # optional baseline
INCLUDE_BASELINE = True                  # compare tuned vs base in the same run
OLLAMA_BASE_URL = "http://localhost:11434"

USE_JUDGE = bool(os.environ.get("PROMPTLENS_API_KEY"))
RUN_TEACHER = False                      # set True only if you want the frontier teacher in the same eval

JUDGE_CFG = {
    "name": "gpt-5.5",
    "provider": "openai_compatible",
    "model": "openai-group/gpt-5.5",
    "base_url": "https://tfy.promptlens.trilogy.com/v1",
    "api_key_env": "PROMPTLENS_API_KEY",
    "max_tokens": 2048,
    "include_temperature": False,
    "token_param": "max_completion_tokens",
}

TEACHER_CFG = {
    "name": "claude-opus-4-8",
    "provider": "openai_compatible",
    "model": "claude-group/claude-opus-4-8",
    "base_url": "https://tfy.promptlens.trilogy.com/v1",
    "api_key_env": "PROMPTLENS_API_KEY",
    "max_tokens": 8192,
}

def ollama_candidate(name, model):
    return {
        "name": name,
        "provider": "ollama",
        "model": model,
        "base_url": OLLAMA_BASE_URL,
        "think": False,
        "keep_alive": "15m",
        "format": "json",
        "max_tokens": 1536,
        "num_ctx": 4096,
    }

candidates = []
if INCLUDE_BASELINE:
    candidates.append(ollama_candidate("qwen3-4b-base", BASE_MODEL_NAME))
candidates.append(ollama_candidate("qwen3-apush-tuned", MODEL_NAME))

models = {"candidates": candidates}
if USE_JUDGE:
    models["judge"] = JUDGE_CFG
    if RUN_TEACHER:
        models["teacher"] = TEACHER_CFG

CONFIG_PATH = ROOT / "eval" / "models.notebook.json"
CONFIG_PATH.write_text(json.dumps(models, indent=2) + "\n", encoding="utf-8")
print(CONFIG_PATH)
print(json.dumps(models, indent=2))
if not USE_JUDGE:
    print("\nNo PROMPTLENS_API_KEY found: eval cells will add --no-judge by default.")

In [ ]:
def run_cmd(cmd, *, check=True, timeout=None):
    print("+", " ".join(str(x) for x in cmd))
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if check and proc.returncode:
        raise RuntimeError(f"command failed with exit {proc.returncode}: {cmd}")
    return proc

run_cmd(["ollama", "list"], check=False)

## Inspect the Registered Ollama Template

This catches the classic transfer bug: a Modelfile/template that inserts thinking mode, unexpected system text, markdown wrappers, or a chat template that differs from the notebook path.

In [ ]:
proc = run_cmd(["ollama", "show", MODEL_NAME, "--modelfile"], check=False)
modelfile_text = proc.stdout or ""
print("\n--- quick flags ---")
checks = {
    "contains_<think>": "<think" in modelfile_text.lower(),
    "contains_/think_suffix": "/think" in modelfile_text.lower(),
    "contains_no_think_suffix": "/no_think" in modelfile_text.lower(),
    "has_TEMPLATE": "TEMPLATE" in modelfile_text,
    "has_SYSTEM": "SYSTEM" in modelfile_text,
    "has_tool_scaffold": "<tools>" in modelfile_text or "tool_call" in modelfile_text,
}
for k, v in checks.items():
    print(f"{k:28} {v}")

if checks["contains_<think>"] or checks["contains_/think_suffix"]:
    print("\nWARNING: template appears to mention thinking. The API call still sends think:false, but inspect this carefully.")
if proc.returncode:
    print("\nCould not inspect model. Make sure Ollama is running and MODEL_NAME is registered.")

## Format-Transfer Smoke Test

This uses the exact repo prompt builder plus the native Ollama provider (`think:false`, `format:'json'`). It prints the raw model response and runs the programmatic gates on parsed items.

In [ ]:
import providers
import checks as programmatic_checks
from prompt_loader import LitmusPrompt, extract_items, canonicalize_item_archetype
from harness import load_sources, DIFFICULTY

prompt = LitmusPrompt.from_file(ROOT / "prompts" / "litmus_generation_prompt.md")
source = load_sources("LITMUS")[0]
requested_archetype = "CAUSE_OF_SOURCE"
system, user = prompt.build(
    source=source["text"],
    attribution=source.get("attribution", ""),
    note="",
    n=1,
    archetypes=requested_archetype,
    difficulty=DIFFICULTY,
    include_fewshot=False,
)

cfg = ollama_candidate("qwen3-apush-tuned", MODEL_NAME)
raw = providers.generate(cfg, system, user, temperature=0.0, role="candidate")
print("--- raw response ---")
print(raw)

print("\n--- format flags ---")
stripped = raw.strip()
format_flags = {
    "starts_with_json_array": stripped.startswith("["),
    "contains_think_tag": "<think" in stripped.lower() or "</think" in stripped.lower(),
    "contains_markdown_fence": "```" in stripped,
    "contains_preface_text_before_json": bool(stripped) and not stripped.startswith(("[", "{")),
}
for k, v in format_flags.items():
    print(f"{k:32} {v}")

items = [canonicalize_item_archetype(it, requested_archetype=requested_archetype) for it in extract_items(raw)]
print(f"\nparsed items: {len(items)}")
for i, item in enumerate(items, start=1):
    prog = programmatic_checks.run_checks(item, source)
    print(f"\nitem {i}")
    print(json.dumps(item, indent=2)[:2500])
    print("programmatic:", json.dumps(prog, indent=2))

## Harness Dry Run

This proves the eval plumbing without touching Ollama or external APIs. The numbers are fake.

In [ ]:
run_cmd([sys.executable, "eval/harness.py", "--dry-run", "--runs", "1", "--limit", "1", "--n", "2", "--out", "results/notebook_dry_run"], timeout=180)

## Smoke Eval

Small, cheap run over two sources. If `USE_JUDGE` is false, this runs programmatic checks only.

In [ ]:
smoke_cmd = [
    sys.executable,
    "eval/harness.py",
    "--models", str(CONFIG_PATH),
    "--runs", "1",
    "--limit", "2",
    "--n", "2",
    "--out", "results/notebook_smoke",
]
if not USE_JUDGE:
    smoke_cmd.append("--no-judge")
run_cmd(smoke_cmd, timeout=1800)

## Full Eval

Flip `RUN_FULL_EVAL` after the smoke test is clean. Local Ollama runs serially, so this can take a while.

In [ ]:
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    full_cmd = [
        sys.executable,
        "eval/harness.py",
        "--models", str(CONFIG_PATH),
        "--runs", "3",
        "--n", "6",
        "--out", "results/notebook_full",
    ]
    if not USE_JUDGE:
        full_cmd.append("--no-judge")
    run_cmd(full_cmd, timeout=None)
else:
    print("RUN_FULL_EVAL is False. Change it to True after the smoke eval passes.")

## Read Results

In [ ]:
RESULT_DIR = ROOT / "results" / ("notebook_full" if RUN_FULL_EVAL else "notebook_smoke")
report_path = RESULT_DIR / "litmus_results.md"
items_path = RESULT_DIR / "litmus_items.jsonl"

if report_path.exists():
    print(report_path)
    print(report_path.read_text(encoding="utf-8")[:6000])
else:
    print("No report yet:", report_path)

In [ ]:
if items_path.exists():
    rows = [json.loads(line) for line in items_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("items:", len(rows))
    cols = ["model", "source_id", "archetype", "near_miss", "key_valid", "answer", "stem"]
    if pd is not None:
        display(pd.DataFrame(rows)[[c for c in cols if c in rows[0]]])
    elif rows:
        print(json.dumps(rows[0], indent=2)[:4000])
else:
    print("No per-item file yet:", items_path)